# 06 — Telephony — Twilio

Webhook parsing, HMAC-SHA1, AMD, DTMF, recording, transfer, ring timeout, status callback, TwiML emission.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. See `examples/notebooks/python/Dockerfile` and `docker-compose.yml` for what it builds.


In [ ]:
# Optional — launch the patter-notebooks Docker stack from this cell.
# Skip this cell to run on your host Python. Idempotent if uncommented.
#
# import _setup
# _setup.start_docker()             # build + up -d, prints http://localhost:8888
# _setup.start_docker(open_url=True)  # …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises Twilio webhook signature verification and carrier construction.


### Twilio signature verification — valid request


In [ ]:
import hmac, hashlib, base64
from twilio.request_validator import RequestValidator
with _setup.cell('twilio_sig_valid', tier=1, env=env) as ok:
    if ok:
        auth_token = 'test_auth_token_32chars_padding___'
        url        = 'https://example.com/webhook/voice'
        params     = {'CallSid': 'CA0000000000000000000000000000a001', 'From': '+15555550100', 'To': '+15555550200'}
        validator  = RequestValidator(auth_token)
        signature  = validator.compute_signature(url, params)
        valid      = validator.validate(url, params, signature)
        print(f'signature: {signature}')
        print(f'valid:     {valid}')
        assert valid, 'signature should be valid'


### Twilio signature verification — tampered request


In [ ]:
from twilio.request_validator import RequestValidator
with _setup.cell('twilio_sig_invalid', tier=1, env=env) as ok:
    if ok:
        auth_token = 'test_auth_token_32chars_padding___'
        url        = 'https://example.com/webhook/voice'
        params     = {'CallSid': 'CA0000000000000000000000000000a001', 'From': '+15555550100'}
        validator  = RequestValidator(auth_token)
        bad_sig    = 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA='
        valid      = validator.validate(url, params, bad_sig)
        print(f'tampered signature valid: {valid}  (expected False)')
        assert not valid, 'tampered signature must be rejected'


### E.164 phone number patterns


In [ ]:
import re
with _setup.cell('e164_patterns', tier=1, env=env) as ok:
    if ok:
        E164_RE = re.compile(r'^\+[1-9]\d{6,14}$')
        cases = [
            ('+15555550100', True),
            ('+442071838750', True),
            ('+393399123456', True),
            ('15555550100',  False),   # missing +
            ('+1',          False),   # too short
            ('not-a-number', False),
        ]
        for number, expected in cases:
            result = bool(E164_RE.match(number))
            status = '✓' if result == expected else '✗'
            print(f'  {status} {number!r:20s} → {result}')
        assert all(bool(E164_RE.match(n)) == e for n, e in cases)


### Twilio carrier construction


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('twilio_carrier', tier=1, env=env) as ok:
    if ok:
        carrier = Twilio(
            account_sid='ACtest00000000000000000000000000',
            auth_token='test_token',
        )
        p = Patter(
            carrier=carrier,
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        lc = p._local_config
        print(f'carrier:  {lc.telephony_provider}')
        print(f'phone:    {lc.phone_number}')
        print(f'webhook:  {lc.webhook_url}')
        assert lc.telephony_provider == 'twilio'


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Tests Twilio call flow including AMD detection. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')
        print(f'  features: AMD + voicemail fallback')


### Live Twilio call with AMD *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime
with _setup.cell('live_twilio_amd', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a Twilio telephony demo agent. Greet the caller and hang up.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        try:
            await p.call(
                env.target_number,
                agent=agent,
                machine_detection=True,
                voicemail_message='You reached Patter demo. Goodbye.',
                first_message='Hello from Patter Twilio demo.',
                ring_timeout=env.max_call_seconds,
            )
            print('✓ Twilio AMD call completed')
        finally:
            _setup.hangup_leftover_calls(env)
